# 1. Transformation Strategy

## 1.1 Objectives

The main objectives of this notebook are:

- load the raw FIPE snapshot without modifying the source;
- validate the expected raw-data assumptions;
- standardize fields with known inconsistencies;
- preserve business semantics identified during profiling;
- validate the transformed dataset;
- export a processed dataset for downstream analytical use.

## 1.2 Design Principles

The transformation process follows these principles:

- raw data must remain immutable;
- transformations must be deterministic and reproducible;
- business rules must be explicit;
- invalid records must be detected rather than silently corrected;
- `valor_centavos` is the canonical monetary field;
- `ano_modelo = NULL` must be preserved when associated with `zero_km = True`;
- the validated snapshot grain must remain unique;
- transformed data must comply with the documented data quality rules.

# 2. Imports and Configuration

In [2]:
import pandas as pd
from fipex.config import RAW_FILE, PROCESSED_FILE
from fipex.io import load_parquet, save_parquet
from fipex.transformations import transform_fipe_data
from fipex.validation import (validate_fipe_data, validate_processed_data)

# 3. Data Loading

## 3.1 Raw Input

The raw source is loaded without applying transformations.

This preserves the original dataset and allows all changes to be explicitly tracked in the transformation pipeline.

In [3]:
df_raw = load_parquet(RAW_FILE)

In [4]:
df_raw.sample(20)

,tipo_veiculo,codigo_fipe,nome_modelo,nome_marca,nome_combustivel,sigla_combustivel,ano_modelo,zero_km,valor_centavos,valor_formatado,mes_referencia,ano_referencia
48876,carro,005095-4,Saveiro TSi 2.0 Mi,VW - VolksWagen,Gasolina,g,1998.0,False,2606100,"R$ 26.061,00",9,2026
41786,moto,814018-9,JH 250 FLY,TRAXX,Gasolina,g,2015.0,False,492000,"R$ 4.920,00",9,2026
19225,carro,014010-4,Civic Sedan LX 1.5/1.6,Honda,Gasolina,g,1994.0,False,1164100,"R$ 11.641,00",9,2026
40093,moto,850015-0,XY 50-Q2 RETRO/JET/BIKE,SHINERAY,Gasolina,g,2017.0,False,414000,"R$ 4.140,00",9,2026
17929,moto,811061-1,CBX 250 TWISTER,HONDA,Gasolina,g,2008.0,False,936100,"R$ 9.361,00",9,2026
39081,caminhão,513192-8,R-420 A 4x2 3-Eixos/ A 6x2 2p (diesel),SCANIA,Diesel,d,2012.0,False,24941700,"R$ 249.417,00",9,2026
24206,carro,018077-7,Sportage LX 2.0 16V/ 2.0 16V Flex Aut.,Kia Motors,Gasolina,g,2016.0,False,7729000,"R$ 77.290,00",9,2026
25827,caminhão,508009-6,VOLARE EXECUTIVO W9 (diesel),MARCOPOLO,Diesel,d,2007.0,False,7980100,"R$ 79.801,00",9,2026
11183,carro,001350-1,UNO WAY Celeb. 1.4 EVO Fire Flex 8V 2p,Fiat,Flex,f,2012.0,False,2996000,"R$ 29.960,00",9,2026
46456,caminhão,516272-6,VM 290 8X4 2p (diesel) (E6),VOLVO,Diesel,d,2023.0,False,48019400,"R$ 480.194,00",9,2026


In [5]:
df_raw.dtypes

tipo_veiculo             str
codigo_fipe              str
nome_modelo              str
nome_marca               str
nome_combustivel         str
sigla_combustivel        str
ano_modelo           float64
zero_km                 bool
valor_centavos         int64
valor_formatado          str
mes_referencia         int32
ano_referencia         int32
dtype: object

In [6]:
df_raw.columns.tolist()

['tipo_veiculo',
 'codigo_fipe',
 'nome_modelo',
 'nome_marca',
 'nome_combustivel',
 'sigla_combustivel',
 'ano_modelo',
 'zero_km',
 'valor_centavos',
 'valor_formatado',
 'mes_referencia',
 'ano_referencia']

# 4. Pre-Transformation Validation

The raw dataset is validated against the documented data quality contract before any transformation is applied.

The validation includes schema, completeness, grain, domain, functional dependency, and monetary consistency checks.

In [7]:
validate_fipe_data(df_raw)

# 5. Data Transformations

The validated raw dataset is transformed using the reusable transformation pipeline defined in `src/fipex/transformations.py`.

The transformation includes string cleanup, brand standardization, dtype normalization, and column reordering.

In [8]:
df = transform_fipe_data(df_raw)

In [9]:
df.head()

,ano_referencia,mes_referencia,tipo_veiculo,codigo_fipe,nome_marca,nome_modelo,ano_modelo,zero_km,nome_combustivel,sigla_combustivel,valor_centavos,valor_formatado
0,2026,9,moto,840015-6,ADLY,ATV 100,2000,False,Gasolina,g,320000,"R$ 3.200,00"
1,2026,9,moto,840015-6,ADLY,ATV 100,2001,False,Gasolina,g,340300,"R$ 3.403,00"
2,2026,9,moto,840015-6,ADLY,ATV 100,2002,False,Gasolina,g,372700,"R$ 3.727,00"
3,2026,9,moto,840014-8,ADLY,ATV 50,2000,False,Gasolina,g,235800,"R$ 2.358,00"
4,2026,9,moto,840014-8,ADLY,ATV 50,2001,False,Gasolina,g,246300,"R$ 2.463,00"


In [10]:
df.dtypes

ano_referencia       int64
mes_referencia       int64
tipo_veiculo           str
codigo_fipe            str
nome_marca             str
nome_modelo            str
ano_modelo           Int64
zero_km               bool
nome_combustivel       str
sigla_combustivel      str
valor_centavos       int64
valor_formatado        str
dtype: object

In [11]:
df.shape

(51012, 12)

# 6. Post-Transformation Validation

After all transformations are applied, the dataset is validated again.

The objective is to confirm that the transformation process did not introduce inconsistencies and that the processed dataset satisfies the expected data quality contract.

In [12]:
validate_processed_data(df_raw, df)

# 7. Transformation Audit

This section summarizes the effects of the transformation pipeline.

The audit is informational only. All validation and quality-gate checks are performed in the post-transformation validation step.

## 7.1 Row Count Summary

In [13]:
raw_rows = len(df_raw)
processed_rows = len(df)

raw_columns = df_raw.shape[1]
processed_columns = df.shape[1]

print(f"Raw rows: {raw_rows:,}")
print(f"Processed rows: {processed_rows:,}")
print(f"Raw columns: {raw_columns}")
print(f"Processed columns: {processed_columns}")

Raw rows: 51,012
Processed rows: 51,012
Raw columns: 12
Processed columns: 12


## 7.2 Column Count Reconciliation

In [14]:
brand_changes = (
    df_raw["nome_marca"]
    != df["nome_marca"]
).sum()

print(f"nome_marca values changed: {brand_changes:,}")

nome_marca values changed: 7,209


In [15]:
brand_change_details = (
    pd.DataFrame({
        "raw": df_raw["nome_marca"],
        "processed": df["nome_marca"],
    })
    .query("raw != processed")
    .drop_duplicates()
    .sort_values(["raw", "processed"])
)

brand_change_details

,raw,processed
16,AGRALE,Agrale
7412,FIAT,Fiat
7428,FORD,Ford
17583,HONDA,Honda
18866,HYUNDAI,Hyundai
26176,MERCEDES-BENZ,Mercedes-Benz
33479,PEUGEOT,Peugeot
40321,SUZUKI,Suzuki
44980,VOLVO,Volvo


## 7.3 Changed Fields

In [16]:
strip_columns = [
    "tipo_veiculo",
    "codigo_fipe",
    "nome_modelo",
    "nome_combustivel",
    "sigla_combustivel",
    "valor_formatado",
]

string_change_counts = {}

for column in strip_columns:
    string_change_counts[column] = (
        df_raw[column] != df[column]
    ).sum()

pd.Series(string_change_counts).sort_values(ascending=False)

nome_modelo          393
tipo_veiculo           0
codigo_fipe            0
nome_combustivel       0
sigla_combustivel      0
valor_formatado        0
dtype: int64

## 7.4 Audit Summary

In [17]:
audit_summary = pd.Series({
    "raw_rows": len(df_raw),
    "processed_rows": len(df),
    "raw_columns": df_raw.shape[1],
    "processed_columns": df.shape[1],
    "brand_values_changed": brand_changes,
    "model_name_values_changed": string_change_counts["nome_modelo"],
})

audit_summary

raw_rows                     51012
processed_rows               51012
raw_columns                     12
processed_columns               12
brand_values_changed          7209
model_name_values_changed      393
dtype: int64

# 8. Processed Data Export

The validated transformed dataset is persisted to the processed data layer in Parquet format.

In [18]:
save_parquet(df, PROCESSED_FILE)

print(f"Processed dataset saved to: {PROCESSED_FILE}")

Processed dataset saved to: E:\VSCODE Files\Projects\01_automotive_market_data_analysis\data\processed\fipex_prices_2026_09.parquet


# 9. Transformation Summary

The FIPE raw snapshot was successfully loaded, validated, transformed, audited, and persisted.

The pipeline:

- preserved the raw source unchanged;
- validated structural and semantic constraints before transformation;
- applied reusable transformation logic from the `src/fipex` package;
- standardized known brand-name inconsistencies;
- removed leading and trailing whitespace from selected textual fields;
- standardized data types;
- preserved zero-kilometer model-year semantics;
- performed complete post-transformation validation;
- summarized row, column, and field-level changes;
- persisted the processed dataset in Parquet format.

The processed dataset is ready for downstream analytical modeling with DuckDB, SQL, and Power BI.